# AMD/NVIDIA Item 2.02 十一份初筛未匹配原件机械复核

复用已备份完整 SEC 原始提交，逐份区分公告类别、定性指引、非典型附件类型与规则遗漏；零新请求。

In [1]:
import json,hashlib,re
from pathlib import Path
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_amd_nvda_sec_guidance_original_probe_20261023';raw=root/'data/snapshots/stage1_amd_nvda_sec_guidance_original_probe_20261023';plan=json.loads((card/'item202_nonmatch_repair_plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(card/'item202_inventory_result.json')==plan['parent_inventory_result_sha256'] and len(plan['exact_nonmatches'])==11 and not (card/'item202_nonmatch_repair_result.json').exists()
manual={
 '0001193125-11-011350':('earnings_qualitative_next_quarter_outlook','AMD revenue outlook says flat to slightly down sequentially; no numeric point/range'),
 '0001193125-12-421061':('preliminary_current_quarter_update','AMD preliminary third-quarter results, not completed-quarter release plus following-quarter outlook'),
 '0000002488-18-000043':('unrelated_accounting_standard_adoption','AMD ASC 606 adoption exhibit, not quarterly earnings/outlook'),
 '0000002488-20-000101':('numeric_forward_parser_false_negative','AMD formal Current Outlook has approximately $2.55 billion plus/minus $100 million for next quarter'),
 '0000002488-21-000009':('numeric_forward_parser_false_negative','AMD formal Current Outlook has approximately $3.2 billion plus/minus $100 million for next quarter'),
 '0000002488-25-000045':('numeric_forward_parser_false_negative','AMD formal Current Outlook has approximately $7.4 billion plus/minus $300 million for next quarter'),
 '0001045810-14-000004':('numeric_forward_parser_false_negative','NVIDIA formal next-quarter outlook has approximately $1.05 billion plus/minus two percent'),
 '0001045810-14-000071':('earnings_no_same_release_numeric_outlook','NVIDIA first-quarter earnings release says additional financial outlook will follow May 8; no same-release numeric next-quarter revenue'),
 '0001045810-19-000004':('preliminary_current_quarter_update','NVIDIA updates current Q4 fiscal 2019 guidance before final quarterly results'),
 '0001045810-19-000168':('numeric_forward_unusual_SEC_exhibit_type','NVIDIA next-quarter earnings press release embedded as SEC EX-95.1 q3fy20pr.htm, rather than EX-99.1'),
 '0001045810-23-000087':('numeric_forward_parser_false_negative','NVIDIA next-quarter outlook has $11.00 billion plus/minus 2 percent')}
assert set(manual)=={x['accession'] for x in plan['exact_nonmatches']}
rows=[]
for x in plan['exact_nonmatches']:
 p=raw/f"item202_{x['accession']}.txt";assert sha(p)==x['raw_sha256'];b=p.read_bytes();head=b[:15000].decode('utf8','replace');assert x['accession'] in head
 sections=[]
 for doc in re.findall(rb'<DOCUMENT>(.*?)</DOCUMENT>',b,re.S|re.I):
  typ=re.search(rb'<TYPE>\s*([^\r\n<]+)',doc,re.I);filename=re.search(rb'<FILENAME>\s*([^\r\n<]+)',doc,re.I);content=re.search(rb'<TEXT>(.*?)</TEXT>',doc,re.S|re.I)
  if typ and content and typ.group(1).strip().upper() in (b'8-K',b'EX-99.1',b'EX99.1',b'EX-99',b'EX-95.1'):
   text=re.sub(r'\s+',' ',BeautifulSoup(content.group(1).decode('utf8','replace'),'html.parser').get_text(' ',strip=True))
   sections.append((typ.group(1).decode(),filename.group(1).decode() if filename else None,text))
 category,reason=manual[x['accession']];needle='revenue'
 if category=='unrelated_accounting_standard_adoption':needle='ASC 606'
 if category=='earnings_no_same_release_numeric_outlook':needle='financial outlook'
 if category=='preliminary_current_quarter_update':needle='preliminary' if x['issuer']=='AMD' else 'updated its financial guidance'
 if category.startswith('numeric_forward'):needle=r'(?:AMD\s+expects\s+revenue|Revenue\s+is\s+expected)' 
 evidence=[]
 for typ,filename,text in sections:
  m=re.search(needle,text,re.I)
  if m:evidence.append({'type':typ,'filename':filename,'excerpt':text[max(0,m.start()-100):m.start()+280]})
 assert evidence,(x['accession'],category)
 if category.startswith('numeric_forward'):
  assert any(re.search(r'\d+(?:\.\d+)?\s*(?:billion|million|percent|%)',e['excerpt'],re.I) for e in evidence),(x['accession'],'numeric support missing')
 rows.append({'issuer':x['issuer'],'filed':x['filed'],'accession':x['accession'],'raw_sha256':x['raw_sha256'],'classification':category,'reason':reason,'evidence':evidence[:2]})
counts={cat:sum(x['classification']==cat for x in rows) for cat in sorted({x['classification'] for x in rows})};result={'status':'eleven_original_Item202_nonmatches_source_semantics_adjudicated_no_outcomes','plan_sha256':sha(card/'item202_nonmatch_repair_plan.json'),'parent_inventory_result_sha256':plan['parent_inventory_result_sha256'],'rows':rows,'category_counts':counts,'numeric_forward_recovered':sum(x['classification'].startswith('numeric_forward') for x in rows),'genuine_no_numeric_next_quarter_outlook':sum(x['classification'] in ('earnings_qualitative_next_quarter_outlook','earnings_no_same_release_numeric_outlook') for x in rows),'nonquarter_or_unrelated':sum(x['classification'] in ('preliminary_current_quarter_update','unrelated_accounting_standard_adoption') for x in rows),'new_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'item202_nonmatch_repair_result.json').write_text(json.dumps(result,indent=2)+'\n');print({k:result[k] for k in ('status','category_counts','numeric_forward_recovered','genuine_no_numeric_next_quarter_outlook','nonquarter_or_unrelated')})


{'status': 'eleven_original_Item202_nonmatches_source_semantics_adjudicated_no_outcomes', 'category_counts': {'earnings_no_same_release_numeric_outlook': 1, 'earnings_qualitative_next_quarter_outlook': 1, 'numeric_forward_parser_false_negative': 5, 'numeric_forward_unusual_SEC_exhibit_type': 1, 'preliminary_current_quarter_update': 2, 'unrelated_accounting_standard_adoption': 1}, 'numeric_forward_recovered': 6, 'genuine_no_numeric_next_quarter_outlook': 2, 'nonquarter_or_unrelated': 3}


## 中文结论

11 份已存原件源语义复核：6 份数字指引规则漏判、2 份季度公告没有同份数字指引、3 份初步/其他公告；零新请求。 完整决定见 `conclusion.md`。